In [ ]:
# feel free to run at start to ensure we are starting with clean cache
!rm -rf data.zip data/ *.parquet

In [ ]:
# environment setup

!pip install pyspark gdown -q
import os
import gdown
from pyspark.sql import SparkSession

# initialize spark session
spark = (SparkSession.builder
  .appName("CS131_Sprint_5")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "8")
  .getOrCreate())

# download data from drive
file_id = "17UR9RRaoKasgV6UPWMoxwlJ7AUTkahTt"
url = f"https://drive.google.com/uc?id={file_id}"
output = "data.zip"

if not os.path.exists("data"):
  print("Fetching data...")
  gdown.download(url, output, quiet=True)
  !unzip -q -o data.zip
  print("Data loaded.")
else:
  print("Data already exists locally.")

Fetching data...
Data loaded.


In [ ]:
# converting csv files to optimized parquet files

print("Converting csv to parquet ...")

matches_raw = spark.read.csv("data/atp_matches.csv", header=True, inferSchema=True)
players_raw = spark.read.csv("data/atp_players.csv", header=True, inferSchema=True)
rankings_raw = spark.read.csv("data/atp_rankings.csv", header=True, inferSchema=True)
qual_raw = spark.read.csv("data/atp_matches_qual_chall.csv", header=True, inferSchema=True)

matches_raw.write.mode("overwrite").parquet(f"data/atp_matches.parquet")
players_raw.write.mode("overwrite").parquet(f"data/atp_players.parquet")
rankings_raw.write.mode("overwrite").parquet(f"data/atp_rankings.parquet")
qual_raw.write.mode("overwrite").parquet(f"data/atp_matches_qual_chall.parquet")

matches_df = spark.read.parquet("data/atp_matches.parquet")
players_df = spark.read.parquet("data/atp_players.parquet")
rankings_df = spark.read.parquet("data/atp_rankings.parquet")
qual_df = spark.read.parquet("data/atp_matches_qual_chall.parquet")

print("Finished converting csv to parquet")

Converting csv to parquet ...
Finished converting csv to parquet


In [ ]:
# Version & Config Checks
print(f"App Name: {spark.conf.get('spark.app.name')}")
print(f"Spark Version: {spark.version}")
master = spark.sparkContext.master
print(f"Master URL: {master}")
partitions = spark.conf.get("spark.sql.shuffle.partitions")
print(f"Shuffle Partitions: {partitions}")

App Name: CS131_Sprint_5
Spark Version: 4.0.2
Master URL: local[*]
Shuffle Partitions: 8


In [ ]:
# data cleaning
from pyspark.sql.functions import col, when, to_date, avg, lit, coalesce

# replace NULL in winner/loser entry columns with 'DA'
# convert tourney_date (string/int) to a date type

matches_df = matches_df.withColumn(
    "winner_entry", when(col("winner_entry").isNull(), "DA").otherwise(col("winner_entry"))
).withColumn(
    "loser_entry", when(col("loser_entry").isNull(), "DA").otherwise(col("loser_entry"))
).withColumn(
    # This tries BOTH common formats. If one fails, it tries the other.
    "tourney_date", coalesce(
        to_date(col("tourney_date").cast("string"), "yyyy-MM-dd"),
        to_date(col("tourney_date").cast("string"), "yyyyMMdd")
    )
).filter((col("w_svpt") > 0) & (col("l_svpt") > 0))
# fill null height values with average heights

# get the average height
avg_height = players_df.select(avg("height")).first()[0]

# update null heights in players table
players_df = players_df.fillna({"height": avg_height})

# Update null heights in matches table
matches_df = matches_df.fillna({"winner_ht": avg_height, "loser_ht": avg_height})

rank_df = (
    rankings_df
    .select(
        col("player").alias("player_id"),
        col("rank").cast("double"),
        col("ranking_date")
    )
    .filter(col("rank").isNotNull())
)

# proof of cleaning
print(f"Filled nulls with average height: {avg_height:.2f}")
print("Matches Nulls (winner_ht):", matches_df.filter(col("winner_ht").isNull()).count())
print("Players Nulls (height):", players_df.filter(col("height").isNull()).count())

Filled nulls with average height: 183.75
Matches Nulls (winner_ht): 0
Players Nulls (height): 0


In [ ]:
# Examples of showing small sample for each file
# matches_df.show(5)
players_df.show(5)
rankings_df.show(5)
qual_df.show(5)

+---------+----------+---------+----+--------+---+------+-----------+
|player_id|name_first|name_last|hand|     dob|ioc|height|wikidata_id|
+---------+----------+---------+----+--------+---+------+-----------+
|   100001|   Gardnar|   Mulloy|   R|19131122|USA|   185|     Q54544|
|   100002|    Pancho|   Segura|   R|19210620|ECU|   168|     Q54581|
|   100003|     Frank|  Sedgman|   R|19271002|AUS|   180|    Q962049|
|   100004|  Giuseppe|    Merlo|   R|19271011|ITA|   183|   Q1258752|
|   100005|   Richard| Gonzalez|   R|19280509|USA|   188|     Q53554|
+---------+----------+---------+----+--------+---+------+-----------+
only showing top 5 rows
+------------+----+------+------+
|ranking_date|rank|player|points|
+------------+----+------+------+
|    20240101|   1|104925| 11245|
|    20240101|   2|207989|  8855|
|    20240101|   3|106421|  7600|
|    20240101|   4|206173|  6490|
|    20240101|   5|126094|  4805|
+------------+----+------+------+
only showing top 5 rows
+----------+----

In [ ]:
# non trivial transform (JOIN)

# join matches with players to get the winner's country/IOC
matches_with_bio = matches_df.join(
    players_df.select("player_id", "hand", "ioc"),
    matches_df.winner_id == players_df.player_id,
    "left"
)

# filter for "upsets" where the winner's rank was significantly lower than the loser's

upsets_df = matches_with_bio.filter(col("winner_rank") > col("loser_rank") + 20)

print("Sample of match upsets (Winner Rank > Loser Rank + 20):")
upsets_df.select("tourney_name", "winner_name", "winner_rank", "loser_name", "loser_rank").show(5)

Sample of match upsets (Winner Rank > Loser Rank + 20):
+------------+----------------+-----------+----------------+----------+
|tourney_name|     winner_name|winner_rank|      loser_name|loser_rank|
+------------+----------------+-----------+----------------+----------+
|    Brisbane| James Duckworth|        116|Yannick Hanfmann|        51|
|    Brisbane|    Rafael Nadal|        672|    Jason Kubler|       102|
|    Brisbane| James Duckworth|        116|        J J Wolf|        53|
|    Brisbane|Yannick Hanfmann|         51| Sebastian Korda|        24|
|    Brisbane| Roman Safiullin|         39|     Ben Shelton|        17|
+------------+----------------+-----------+----------------+----------+
only showing top 5 rows


In [ ]:
# non trivial transform (groupBy)

from pyspark.sql.functions import avg, count, round

# group by surface and player to see dominance patterns
surface_dominance = matches_df.groupBy("winner_name", "surface").agg(
    count("match_num").alias("wins"),
    round(avg("w_ace"), 2).alias("avg_aces_per_win"),
    round(avg("w_1stWon"), 2).alias("avg_1st_serve_pts")
).filter(col("wins") > 1).orderBy(col("avg_aces_per_win").desc())

print("Surface Dominance (Players with >1 win on a surface):")
surface_dominance.show(10)

Surface Dominance (Players with >1 win on a surface):
+-----------------+-------+----+----------------+-----------------+
|      winner_name|surface|wins|avg_aces_per_win|avg_1st_serve_pts|
+-----------------+-------+----+----------------+-----------------+
|Joachim Johansson|  Grass|   6|            27.5|             65.5|
|       John Isner|  Grass|  56|           24.16|            53.89|
|  Robert Kendrick|   Clay|   2|            23.0|             71.0|
|     Ivo Karlovic| Carpet|   6|           22.83|             49.5|
|     Ivo Karlovic|  Grass|  74|           22.57|            47.88|
|      Mirza Basic| Carpet|   2|            22.5|             58.5|
| Goran Ivanisevic|  Grass|  62|           21.95|            60.98|
|     Nick Kyrgios|  Grass|  34|           21.85|            57.56|
|     Milos Raonic|  Grass|  51|            21.8|            48.94|
|  Alexander Waske| Carpet|   3|           21.33|            49.33|
+-----------------+-------+----+----------------+-------------

In [ ]:
# Define the path to your data directory
path = "data/"

In [ ]:
# =========================================================
# ENG2 STARTS HERE
# Task: Frequency Table 1 using PySpark
# =========================================================


# =========================================================
# ENG2 - Step 1: Inspect schema
# =========================================================
matches_df.printSchema()


# =========================================================
# ENG2 - Step 2: Preview surface column
# =========================================================
matches_df.select("surface").show(20, truncate=False)


# =========================================================
# ENG2 - Step 3: Import functions
# =========================================================
from pyspark.sql.functions import col, trim, count


# =========================================================
# ENG2 - Step 4: Check null / blank values
# =========================================================
bad_surface_count = matches_df.where(
    col("surface").isNull() | (trim(col("surface")) == "")
).count()

print("Missing/blank surface rows:", bad_surface_count)


# =========================================================
# ENG2 - Step 5: Create frequency table
# =========================================================
surface_freq_df = (
    matches_df
    .where(col("surface").isNotNull() & (trim(col("surface")) != ""))
    .groupBy("surface")
    .agg(count("*").alias("frequency"))
    .orderBy(col("frequency").desc())
)


# =========================================================
# ENG2 - Step 6: Show result
# =========================================================
surface_freq_df.show()


# =========================================================
# ENG2 - Step 7: Sanity check
# =========================================================
valid_surface_rows = matches_df.where(
    col("surface").isNotNull() & (trim(col("surface")) != "")
).count()

print("Valid surface rows:", valid_surface_rows)

surface_freq_df.selectExpr("sum(frequency) as total_counted").show()


# =========================================================
# ENG2 - Step 8: Save as Parquet
# =========================================================
surface_freq_df.write.mode("overwrite").parquet(f"{path}surface_frequency_table.parquet")


# =========================================================
# ENG2 - Step 9: Save as CSV
# =========================================================
surface_freq_df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{path}surface_frequency_table_csv")


# =========================================================
# ENG2 COMPLETED
# =========================================================

root
 |-- tourney_id: string (nullable = true)
 |-- tourney_name: string (nullable = true)
 |-- surface: string (nullable = true)
 |-- draw_size: integer (nullable = true)
 |-- tourney_level: string (nullable = true)
 |-- tourney_date: date (nullable = true)
 |-- match_num: integer (nullable = true)
 |-- winner_id: integer (nullable = true)
 |-- winner_seed: string (nullable = true)
 |-- winner_entry: string (nullable = true)
 |-- winner_name: string (nullable = true)
 |-- winner_hand: string (nullable = true)
 |-- winner_ht: integer (nullable = true)
 |-- winner_ioc: string (nullable = true)
 |-- winner_age: double (nullable = true)
 |-- loser_id: integer (nullable = true)
 |-- loser_seed: string (nullable = true)
 |-- loser_entry: string (nullable = true)
 |-- loser_name: string (nullable = true)
 |-- loser_hand: string (nullable = true)
 |-- loser_ht: integer (nullable = true)
 |-- loser_ioc: string (nullable = true)
 |-- loser_age: double (nullable = true)
 |-- score: string (nulla

In [ ]:
# =========================================================
# ENG3 STARTS HERE
# Task: Top 100 Players by Wins
# =========================================================

from pyspark.sql.functions import col, count

# Step 1: Check schema
matches_df.printSchema()

# Step 2: Create Top-100 players by wins
top_players_df = (
    matches_df
    .where(col("winner_id").isNotNull())
    .groupBy("winner_id")
    .agg(count("*").alias("win_count"))
    .orderBy(col("win_count").desc())
    .limit(100)
)

# Step 3: Show result
top_players_df.show()

# Step 4: Save as Parquet
top_players_df.write.mode("overwrite").parquet(f"{path}top_100_players.parquet")

# Step 5: Save as CSV
top_players_df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{path}top_100_players_csv")

# =========================================================
# ENG3 COMPLETED
# =========================================================

root
 |-- tourney_id: string (nullable = true)
 |-- tourney_name: string (nullable = true)
 |-- surface: string (nullable = true)
 |-- draw_size: integer (nullable = true)
 |-- tourney_level: string (nullable = true)
 |-- tourney_date: date (nullable = true)
 |-- match_num: integer (nullable = true)
 |-- winner_id: integer (nullable = true)
 |-- winner_seed: string (nullable = true)
 |-- winner_entry: string (nullable = true)
 |-- winner_name: string (nullable = true)
 |-- winner_hand: string (nullable = true)
 |-- winner_ht: integer (nullable = true)
 |-- winner_ioc: string (nullable = true)
 |-- winner_age: double (nullable = true)
 |-- loser_id: integer (nullable = true)
 |-- loser_seed: string (nullable = true)
 |-- loser_entry: string (nullable = true)
 |-- loser_name: string (nullable = true)
 |-- loser_hand: string (nullable = true)
 |-- loser_ht: integer (nullable = true)
 |-- loser_ioc: string (nullable = true)
 |-- loser_age: double (nullable = true)
 |-- score: string (nulla

In [ ]:
from pyspark.sql.functions import sum, round, avg, min, substring
winners = matches_df.select(
    col("winner_id").alias("player_id"),
    col("w_1stIn").cast("double").alias("first_in"),
    col("w_svpt").cast("double").alias("total_sv"),
    lit(1).alias("win"),
    lit(0).alias("loss")
)

losers = matches_df.select(
    col("loser_id").alias("player_id"),
    col("l_1stIn").cast("double").alias("first_in"),
    col("l_svpt").cast("double").alias("total_sv"),
    lit(0).alias("win"),
    lit(1).alias("loss")
)

player_matches = winners.unionByName(losers)

player_matches = player_matches.filter(
    (col("first_in").isNotNull()) &
    (col("total_sv").isNotNull()) &
    (col("total_sv") > 0)
)

player_stats = (
    player_matches
    .groupBy("player_id")
    .agg(
        sum("first_in").alias("first_in"),
        sum("total_sv").alias("total_sv"),
        count("*").alias("matches"),
        sum("win").alias("wins"),
        sum("loss").alias("losses")
    )
)

rank_stats = (
    rank_df
    .groupBy("player_id")
    .agg(
        round(avg("rank")).alias("avg_rank"),
        min("rank").alias("best_rank")
    )
)

player_eff = (
    player_stats
    .filter(col("matches") > 10)
    .withColumn("first_serve_eff", round(col("first_in") / col("total_sv")* 100, 2))
    .withColumn("win_rate", round(col("wins") / col("matches")* 100, 2))
    .withColumn(
        "scout_rank",
        when(col("first_serve_eff") > 0.75, "ELITE_CONSISTENT")
        .when(col("first_serve_eff") > 0.60, "PRO-READY")
        .otherwise("DEVELOPMENTAL")
    )
)

elite_summary = (
    player_eff
    .filter((col("scout_rank") == "ELITE_CONSISTENT"))
    .join(players_df, "player_id", "left")
    .join(rank_stats, "player_id", "left")
    .withColumn("birth_year", substring(col("dob"), 1, 4))
    .orderBy(col("win_rate").desc())
    .select(
        "name_first",
        "name_last",
        "matches",
        "first_serve_eff",
        "win_rate",
        "hand",
        "height",
        "ioc",
        "birth_year",
        "avg_rank",
        "best_rank"
    )


)


elite_summary.show(10)


+----------+---------+-------+---------------+--------+----+------+---+----------+--------+---------+
|name_first|name_last|matches|first_serve_eff|win_rate|hand|height|ioc|birth_year|avg_rank|best_rank|
+----------+---------+-------+---------------+--------+----+------+---+----------+--------+---------+
|     Novak| Djokovic|   1274|          64.84|   83.83|   R|   188|SRB|      1987|    51.0|      1.0|
|    Rafael|    Nadal|   1271|          67.99|   82.69|   L|   185|ESP|      1986|    63.0|      1.0|
|     Roger|  Federer|   1458|          62.13|   81.96|   R|   185|SUI|      1981|    46.0|      1.0|
|      Pete|  Sampras|    789|          59.47|   79.97|   R|   185|USA|      1971|    20.0|      1.0|
|    Carlos|  Alcaraz|    219|          65.28|   78.54|   R|   185|ESP|      2003|   338.0|      1.0|
|     Andre|   Agassi|    849|          62.62|   76.56|   R|   180|USA|      1970|    42.0|      1.0|
|   Marcelo|  Arevalo|     12|          62.46|    75.0|   R|   191|ESA|      1990|